# APRIORI

**Status:** Em andamento · **Área:** ideb-regiao-norte

Versão mais abrangente de IDEB com análise, associação e anomalias; inclui URL solta em código, experimentos PyCaret e erro salvo de estilização.

Este arquivo está identificado como **Em andamento**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


### ** PÓS-GRADUAÇÃO DATA SCIENCE E INTELIGÊNCIA ARTIFICIAL**

![Captura de tela 2025-04-10 120950.png](assets/figura-01.png)

### **Trabalho de Aprendizado Não Supervisionado**: ANÁLISE DA PERFORMANCE DAS ESCOLAS PÚBLICAS BRASILEIRAS NO IDEB (2005-2023)

### **PROFESSOR:** ROGERIO LOPES

### **ALUNO:**

* Robert de Jesus Araujo
* Daniel Figueredo Rodrigues
* Elivânio Geraldo
* Weslley Barros



# 1. Entendimento do Negócio

Este projeto tem como objetivo identificar perfis distintos de escolas públicas brasileiras com base na sua evolução de desempenho entre 2005 e 2023, utilizando variáveis como:

- IDEB (Índice de Desenvolvimento da Educação Básica)
- Taxa de Aprovação
- Desempenho no SAEB (Sistema de Avaliação da Educação Básica)

A clusterização poderá apoiar políticas públicas e identificar redes de ensino com melhores práticas.

**Motivação do Projeto**
Este projeto visa identificar perfis distintos de escolas públicas brasileiras com base na evolução de indicadores educacionais entre 2005 e 2023. Esse diagnóstico é essencial para:
- **Direcionar Políticas Públicas:** Identificar regiões e redes com baixo desempenho permite alocar recursos de forma mais eficiente.
- **Compartilhar Boas Práticas:** Escolas com resultados positivos podem servir como modelo para outras.
- **Planejar Investimentos:** Políticas baseadas em dados promovem um uso mais estratégico dos recursos.




# Importações



Todas as importações que serão utilizadas no decorrer do codigo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Referência dos dados

https://drive.google.com/file/d/1lXx9FbMhDFvkHwudLCDOhCba3meUQlbB/view?usp=drive_link

In [ ]:
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Estilo dos gráficos
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")
pd.set_option("display.max_columns", None)  # Mostrar todas as colunas no df


# Leitura

In [ ]:
def descompactar_arquivos(nome_arquivo_zip, destino):
  """
  Descompacta um arquivo zip.

  Args:
    divulgacao_anos_finais_municipios_2023(1).zip: O nome do arquivo zip a ser descompactado.
    destino: O diretório onde os arquivos serão extraídos.
  """
  try:
    with zipfile.ZipFile(nome_arquivo_zip, 'r') as zip_ref:
      zip_ref.extractall(destino)
    print(f"Arquivo {nome_arquivo_zip} descompactado com sucesso para {destino}")
  except FileNotFoundError:
    print(f"Arquivo {nome_arquivo_zip} não encontrado.")
  except Exception as e:
    print(f"Erro ao descompactar o arquivo: {e}")

In [ ]:
zip_prefeitos = "/content/divulgacao_anos_finais_municipios_2023.zip"
descompactar_arquivos( zip_prefeitos, './file_ideb')

In [ ]:
df_base = pd.read_excel("/content/file_ideb/divulgacao_anos_finais_municipios_2023/divulgacao_anos_finais_municipios_2023.xlsx",skiprows=9)

In [ ]:
df_base.head(5)

# Etapa 2 – Entendimento dos Dados (Data Understanding)

Nesta etapa, realizamos o carregamento e a exploração inicial do conjunto de dados disponibilizado pelo INEP, com foco em compreender sua estrutura, qualidade e potencial analítico.

Fonte dos dados
Os dados utilizados foram obtidos a partir de bases públicas do Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP), contendo informações consolidadas por município, estado e rede de ensino, em diferentes anos. Entre os principais indicadores disponíveis estão:

 - IDEB (Índice de Desenvolvimento da Educação Básica)
 - Taxa de Aprovação por série e rede de ensino
 - Notas padronizadas do SAEB (Sistema de Avaliação da Educação Básica) em Língua Portuguesa e Matemática

# Limpeza de Dados

In [ ]:
# Descrição estatística inicial
print("Resumo estatístico das variáveis principais:")
print(df_base.describe())

# Verificar valores nulos e justificar a abordagem
print("\nValores nulos antes do preenchimento:")
print(df_base.isnull().sum())

# Justificar escolha de preenchimento
df_base.fillna(df_base.median(numeric_only=True), inplace=True)  # Mediana é robusta a outliers
print("\nValores nulos após preenchimento:")
print(df_base.isnull().sum())


# Etapa 3 – Preparação dos Dados (Data Preparation)
Nesta etapa do processo CRISP-DM, **iniciamos o tratamento dos dados com foco na região Norte do Brasil**, conforme definido para o nosso grupo. A preparação dos dados envolve a seleção, limpeza, padronização e transformação das variáveis que serão utilizadas na análise.

### Foco regional
O conjunto de dados original foi filtrado para incluir apenas os estados pertencentes à **Região Norte**, utilizando as colunas como referência para identificar os registros correspondentes.

### Colunas:
 - colunas_uteis = [
    'SG_UF', 'NO_MUNICIPIO', 'REDE',
    'VL_OBSERVADO_2005', 'VL_OBSERVADO_2023',
    'VL_NOTA_MATEMATICA_2005', 'VL_NOTA_MATEMATICA_2023',
    'VL_NOTA_PORTUGUES_2005', 'VL_NOTA_PORTUGUES_2023',
    'VL_APROVACAO_2005_1', 'VL_APROVACAO_2005_2', 'VL_APROVACAO_2005_3', 'VL_APROVACAO_2005_4',
    'VL_APROVACAO_2023_1', 'VL_APROVACAO_2023_2', 'VL_APROVACAO_2023_3', 'VL_APROVACAO_2023_4'
]

```


# Filtragem da Região Norte/ redes válidas / Notas 05 e 23 do 1 ao 4 ano

In [ ]:
# Filtro da Região Norte
estados_norte = ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO']
df_base = df_base[df_base['SG_UF'].isin(estados_norte)]

# Redes válidas
df_base = df_base[df_base['REDE'].isin(['Municipal', 'Estadual', 'Federal'])]

# Seleção das colunas realmente existentes (anos iniciais)
colunas_uteis = [
    'SG_UF', 'NO_MUNICIPIO', 'REDE',
    'VL_OBSERVADO_2005', 'VL_OBSERVADO_2023',
    'VL_NOTA_MATEMATICA_2005', 'VL_NOTA_MATEMATICA_2023',
    'VL_NOTA_PORTUGUES_2005', 'VL_NOTA_PORTUGUES_2023',
    'VL_APROVACAO_2005_1', 'VL_APROVACAO_2005_2', 'VL_APROVACAO_2005_3', 'VL_APROVACAO_2005_4',
    'VL_APROVACAO_2023_1', 'VL_APROVACAO_2023_2', 'VL_APROVACAO_2023_3', 'VL_APROVACAO_2023_4'
]

# Atualiza o df_base com apenas as colunas necessárias
df_base = df_base[colunas_uteis].copy()


In [ ]:
# Tratamento para erros na conversão numérica
#for col in colunas_numericas:
#    try:
#        df_base[col] = pd.to_numeric(df_base[col], errors='coerce')
#    except Exception as e:
#        print(f"Erro ao processar a coluna {col}: {e}")

# 4. Modelagem (Modeling)
Nesta etapa, aplicamos técnicas de agrupamento (clustering) para segmentar os municípios da região Norte com base na evolução dos indicadores educacionais entre 2005 e 2023. Utilizamos o algoritmo KMeans, com os dados previamente normalizados.

**Por que KMeans?**
O KMeans é ideal para agrupamentos com formatos compactos e bem definidos, como neste caso em que utilizamos dados normalizados. No entanto, ele não lida bem com outliers, o que justifica a comparação com outros algoritmos (DBSCAN, Hierárquico).


Seleção de variáveis para modelagem
As variáveis utilizadas foram as diferenças (deltas) entre os valores de 2023 e 2005:

- `DELTA_IDEB`: Variação do IDEB
- `DELTA_APROVACAO`: Variação da Taxa de Aprovação
- `DELTA_LP`: Variação da nota SAEB em Língua Portuguesa (dados nulos)
- `DELTA_MT`: Variação da nota SAEB em Matemática (dados nulos)

Essas variáveis refletem a evolução do desempenho educacional ao longo dos anos.

Normalização dos dados
Para que as variáveis tenham a mesma escala e contribuam igualmente na formação dos clusters, aplicamos normalização via z-score, utilizando o StandardScaler.



# Deltas

In [ ]:
# Força conversão para número apenas nas colunas numéricas
colunas_numericas = [col for col in df_base.columns if 'VL_' in col or 'APROVACAO' in col]
df_base[colunas_numericas] = df_base[colunas_numericas].apply(pd.to_numeric, errors='coerce')


In [ ]:
# Deltas de evolução entre 2005 e 2023
df_base['DELTA_IDEB'] = df_base['VL_OBSERVADO_2023'] - df_base['VL_OBSERVADO_2005']
df_base['DELTA_MT'] = df_base['VL_NOTA_MATEMATICA_2023'] - df_base['VL_NOTA_MATEMATICA_2005']
df_base['DELTA_LP'] = df_base['VL_NOTA_PORTUGUES_2023'] - df_base['VL_NOTA_PORTUGUES_2005']

# Média das taxas de aprovação dos anos iniciais (1º ao 4º)
df_base['APROVACAO_2005_MEDIA'] = df_base[[f'VL_APROVACAO_2005_{i}' for i in range(1, 5)]].mean(axis=1)
df_base['APROVACAO_2023_MEDIA'] = df_base[[f'VL_APROVACAO_2023_{i}' for i in range(1, 5)]].mean(axis=1)
df_base['DELTA_APROVACAO'] = df_base['APROVACAO_2023_MEDIA'] - df_base['APROVACAO_2005_MEDIA']


# Normalização de valores Deltas

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Selecionando as colunas a serem normalizadas
colunas_para_normalizar = ['DELTA_IDEB', 'DELTA_MT', 'DELTA_LP', 'DELTA_APROVACAO']

# 2. Inicializando o StandardScaler para normalizar os dados
scaler = StandardScaler()

X_normalizado = scaler.fit_transform(df_base[colunas_para_normalizar])



# Visualizações Primarias

In [ ]:
# 1. Distribuição das taxas de aprovação 2005 vs 2023
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_base[['VL_APROVACAO_2005_1', 'VL_APROVACAO_2023_1']])
plt.title("Distribuição das Taxas de Aprovação (2005 vs 2023)")
plt.ylabel("Taxa de Aprovação")
plt.xticks([0, 1], ['2005', '2023'])
plt.show()

In [ ]:
# 2. Distribuição das notas SAEB Matemática (2005 vs 2023)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_base[['VL_NOTA_MATEMATICA_2005', 'VL_NOTA_MATEMATICA_2023']])
plt.title("Distribuição das Notas de Matemática (2005 vs 2023)")
plt.ylabel("Nota de Matemática")
plt.xticks([0, 1], ['2005', '2023'])
plt.show()

In [ ]:
# 3. Distribuição das notas SAEB Português (2005 vs 2023)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_base[['VL_NOTA_PORTUGUES_2005', 'VL_NOTA_PORTUGUES_2023']])
plt.title("Distribuição das Notas de Português (2005 vs 2023)")
plt.ylabel("Nota de Português")
plt.xticks([0, 1], ['2005', '2023'])
plt.show()

In [ ]:
# 4. Evolução do IDEB de 2005 a 2023
plt.figure(figsize=(12, 6))
sns.lineplot(x=['2005', '2023'], y=[df_base['VL_OBSERVADO_2005'].mean(), df_base['VL_OBSERVADO_2023'].mean()])
plt.title("Evolução do IDEB (2005 vs 2023)")
plt.ylabel("Índice IDEB")
plt.show()

# Clusterizações (Kmeans, DBscan, Hierarquico)

## Metodo Cotovelo


In [ ]:
# Verificando valores nulos no X_normalizado
print(np.isnan(X_normalizado).sum())  # Verifica quantos NaN existem

# Preenchendo valores nulos com a média das colunas
X_normalizado = np.nan_to_num(X_normalizado, nan=np.nanmean(X_normalizado, axis=0))


In [ ]:
# 1. Método do Cotovelo para determinar o número ideal de clusters (de 1 a 10)
wcss = []  # Lista para armazenar os valores do WCSS
for i in range(1, 11):  # Tentando clusters de 1 a 10
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(X_normalizado)  # Usando o X_normalizado para o ajuste
    wcss.append(kmeans.inertia_)  # Calculando a inércia (WCSS) e armazenando

# 2. Gráfico do Método do Cotovelo
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), wcss)  # Gráfico de WCSS por número de clusters
plt.title('Método do Cotovelo para KMeans')
plt.xlabel('Número de Clusters')
plt.ylabel('WCSS (Soma das distâncias quadráticas)')
plt.xticks(range(1, 11))
plt.grid(True)
plt.show()


## Kmeans

In [ ]:
# Verificar os tipos de dados e garantir que são numéricos
print(df_base[colunas_para_normalizar].dtypes)

# Verificar se há valores nulos (NaN)
print(df_base[colunas_para_normalizar].isnull().sum())

# Caso existam NaNs, podemos preencher com a média:
df_base[colunas_para_normalizar] = df_base[colunas_para_normalizar].fillna(df_base[colunas_para_normalizar].mean())

# Normalizar os dados novamente, caso tenha preenchido NaN
scaler = StandardScaler()
X_normalizado = scaler.fit_transform(df_base[colunas_para_normalizar])

# Agora, rodamos o KMeans com os dados limpos e normalizados
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300, n_init=10, random_state=42)
y_kmeans = kmeans.fit_predict(X_normalizado)


In [ ]:
!pip install -q virtualenv
!virtualenv myenv
!source myenv/bin/activate

In [ ]:
# Instação do pacote pycaret com variação [] para versão full
!pip install -q pycaret[full]

In [ ]:
# verificar instalação do pycaret
import pycaret
print(pycaret.__version__)

In [ ]:
# Exportar o dataset tratado do Colab
df_base.to_csv('df_tratado.csv', index=False)

In [ ]:
df = pd.read_csv('df_tratado.csv')
df = df.dropna()

print(df)

In [ ]:
print(df.isnull().sum())  # Mostra o número de nulos por coluna
print(np.isnan(X_normalizado).sum())  # Verifica se o array normalizado contém NaN

In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True)  # Substitui NaN pela mediana

In [ ]:
from pycaret.clustering import *
import pandas as pd

# Inicia o setup do PyCaret
clust_setup = setup(
    data=df,
    session_id=123,  # Define uma seed para reproducibilidade
    normalize=True,  # Habilita a normalização dos dados
    ignore_features=['SG_UF', 'NO_MUNICIPIO', 'REDE'], #Ignorar colunas
    numeric_features=['VL_OBSERVADO_2005', 'VL_OBSERVADO_2023',
                      'VL_NOTA_MATEMATICA_2005', 'VL_NOTA_MATEMATICA_2023',
                      'VL_NOTA_PORTUGUES_2005', 'VL_NOTA_PORTUGUES_2023',
                      # ... adicione outras colunas relevantes
                     ], preprocess=False #Define as colunas numericas

)

# Cria o modelo KMeans com o número ideal de clusters (ajuste conforme necessário)
kmeans_model = create_model('kmeans')


# Avalia o modelo e plota o gráfico do cotovelo automaticamente
evaluate_model(kmeans_model)

# Atribui os clusters ao DataFrame original
clustered_df = assign_model(kmeans_model)

# Exibe o DataFrame com os clusters atribuídos
print(clustered_df.head())

In [ ]:
df_base.head(5)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Verificando os tipos de dados e valores nulos
print(df_base[colunas_para_normalizar].dtypes)
print(df_base[colunas_para_normalizar].isnull().sum())  # Verificando valores nulos

# Preenchendo valores nulos com a média
df_base[colunas_para_normalizar] = df_base[colunas_para_normalizar].fillna(df_base[colunas_para_normalizar].mean())

# Normalizando os dados novamente, se necessário
scaler = StandardScaler()
X_normalizado = scaler.fit_transform(df_base[colunas_para_normalizar])

# Rodando o KMeans com o número de clusters desejado
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300, n_init=10, random_state=42)
y_kmeans = kmeans.fit_predict(X_normalizado)  # Previsões dos clusters

# Plotando os resultados do método do cotovelo (se necessário, para determinar K)
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(X_normalizado)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), wcss)
plt.title('Método do Cotovelo')
plt.xlabel('Número de Clusters')
plt.ylabel('WCSS')
plt.show()

# Plotando os clusters em 2D
plt.figure(figsize=(8, 6))
plt.scatter(X_normalizado[:, 0], X_normalizado[:, 1], c=y_kmeans, cmap='viridis')
plt.title('Clusters KMeans')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()


**Interpretação do Método do Cotovelo**
O gráfico sugere que o melhor valor de *k* é 3, pois a redução na inércia diminui significativamente a partir desse ponto, indicando clusters bem definidos.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

silhouette_avg = silhouette_score(X_normalizado, y_kmeans)
davies_bouldin = davies_bouldin_score(X_normalizado, y_kmeans)

print(f"Coeficiente de Silhueta: {silhouette_avg}")
print(f"Índice Davies-Bouldin: {davies_bouldin}")

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_normalizado)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_kmeans, cmap='viridis')
plt.title("Visualização dos Clusters com PCA")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.colorbar()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Selecionando as colunas para análise
colunas_para_radar = ['DELTA_IDEB', 'DELTA_MT', 'DELTA_LP', 'DELTA_APROVACAO']

# Definir o número de clusters (já sabemos que são 3 clusters no KMeans)
n_clusters = 3

# Definir a quantidade de colunas (para colocarmos os gráficos lado a lado)
fig, axs = plt.subplots(1, n_clusters, figsize=(18, 6), subplot_kw=dict(polar=True))

# Criar o gráfico de radar para cada cluster
for i in range(n_clusters):
    # Filtrando os dados para o cluster i
    cluster_data = df_base[y_kmeans == i][colunas_para_radar].mean()

    # Criando o gráfico de radar
    categories = cluster_data.index
    values = cluster_data.values

    # Número de variáveis
    num_vars = len(categories)

    # Angulação de cada eixo do gráfico
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

    # O gráfico será um círculo, então fechamos a forma (voltamos ao ponto de origem)
    values = np.concatenate((values, [values[0]]))
    angles += angles[:1]

    # Criando o gráfico de radar e ajustando o título
    axs[i].fill(angles, values, color='blue', alpha=0.25)
    axs[i].plot(angles, values, color='blue', linewidth=2)  # Linhas do gráfico
    axs[i].set_yticklabels([])  # Remove os rótulos do eixo Y
    axs[i].set_xticks(angles[:-1])
    axs[i].set_xticklabels(categories, fontsize=12)
    axs[i].set_title(f"Radar Chart para Cluster {i+1} - Persona {i+1}", size=14, color='blue', fontweight='bold')

# Ajustando o layout dos gráficos
plt.tight_layout()
plt.show()


DELTA_IDEB: Variação do IDEB
DELTA_APROVACAO: Variação da Taxa de Aprovação
DELTA_LP: Variação da nota SAEB em Língua Portuguesa (dados nulos)
DELTA_MT: Variação da nota SAEB em Matemática (dados nulos)

## Dbscan

In [ ]:
from sklearn.cluster import DBSCAN

# Rodando o DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
y_dbscan = dbscan.fit_predict(X_normalizado)

# Visualizando os clusters DBSCAN
plt.figure(figsize=(8, 6))
plt.scatter(X_normalizado[:, 0], X_normalizado[:, 1], c=y_dbscan, cmap='viridis')
plt.title('Clusters DBSCAN')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN

# Inicializando DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)

# Ajustando o DBSCAN
y_dbscan = dbscan.fit_predict(X_normalizado)

# Visualizando as anomalias
plt.figure(figsize=(8, 6))
plt.scatter(X_normalizado[:, 0], X_normalizado[:, 1], c=y_dbscan, cmap='coolwarm')
plt.title('Detecção de Anomalias com DBSCAN')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()


## Hierarquico

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Rodando o Cluster Hierárquico
linked = linkage(X_normalizado, 'ward')

# Gerando o dendrograma
plt.figure(figsize=(10, 7))
dendrogram(linked)
plt.title('Dendrograma - Cluster Hierárquico')
plt.xlabel('Índices dos dados')
plt.ylabel('Distância')
plt.show()


In [ ]:
from sklearn.ensemble import IsolationForest

# Inicializando o modelo de Isolation Forest
model_iforest = IsolationForest(n_estimators=100, contamination='auto', random_state=42)

# Ajustando o modelo nos dados normalizados
model_iforest.fit(X_normalizado)

# Prevendo as anomalias (1 é normal, -1 é anômalo)
y_anomaly = model_iforest.predict(X_normalizado)

# Visualizando as anomalias em um gráfico de dispersão
plt.figure(figsize=(8, 6))
plt.scatter(X_normalizado[:, 0], X_normalizado[:, 1], c=y_anomaly, cmap='coolwarm', marker='o')
plt.title('Detecção de Anomalias com Isolation Forest')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()


# Apriori

O Apriori é usado para identificar padrões frequentes nos dados. Nesse contexto, ele permite analisar a interdependência entre variações no IDEB, taxas de aprovação e notas SAEB, fornecendo insights que podem orientar políticas educacionais.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
import pandas as pd
from IPython.display import display

# 🔎 Selecionando e copiando apenas as colunas relevantes
df_apriori = df_base[['DELTA_IDEB', 'DELTA_MT', 'DELTA_LP', 'DELTA_APROVACAO']].copy()

# 🧼 Convertendo todas as colunas numéricas para garantir que nada seja string
df_apriori = df_apriori.apply(pd.to_numeric, errors='coerce')

# 🔁 Binarizando os dados (1 se aumento, 0 se não aumentou ou nulo)
df_apriori = df_apriori.applymap(lambda x: 1 if x > 0 else 0)


In [ ]:

# ⚙️ Rodando o Apriori com suporte mínimo de 10%
frequent_itemsets = apriori(df_apriori, min_support=0.1, use_colnames=True)

# 📈 Gerando regras de associação com base no Lift
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# 🌐 Traduzindo colunas para o português
rules.rename(columns={
    'antecedents': 'Antecedentes',
    'consequents': 'Consequentes',
    'support': 'Suporte',
    'confidence': 'Confiança',
    'lift': 'Lift',
    'leverage': 'Alavancagem',
    'conviction': 'Convicção'
}, inplace=True)

# ✨ Convertendo os frozensets para frases legíveis
rules['Antecedentes'] = rules['Antecedentes'].apply(lambda x: "Se " + ", ".join(sorted(x)) if isinstance(x, frozenset) else x)
rules['Consequentes'] = rules['Consequentes'].apply(lambda x: "então " + ", ".join(sorted(x)) if isinstance(x, frozenset) else x)


In [ ]:

# 🔍 Filtrando as melhores regras
regras_filtradas = rules[
    (rules['Lift'] > 1.0) &
    (rules['Confiança'] > 0.6) &
    (rules['Suporte'] > 0.1)
]

# 📊 Ordenando as regras filtradas do maior para o menor
regras_ordenadas = regras_filtradas.sort_values(
    by=['Suporte', 'Consequentes', 'Confiança', 'Lift'],
    ascending=[False, False, False, False]
)

In [ ]:

# ✅ Exibindo a tabela
display(
    regras_ordenadas[['Antecedentes', 'Consequentes', 'Suporte', 'Confiança', 'Lift']]
    .style
    .format({
        'Suporte': '{:.2%}',
        'Confiança': '{:.2%}',
        'Lift': '{:.2f}'
    })
    .set_caption("📋 Regras de Associação (Lift ≥ 1.0 e Confiança ≥ 60%)")
)


## Análise das Regras de Associação

### 1. Sobre os dados
- As regras forma filtradas com:

    Lift > 1 (indicando associação interessante),

    Confiança > 60%,

    Suporte > 10%.

- Os itens analisados são variações (DELTA) de indicadores educacionais: IDEB, MT, LP e APROVAÇÃO.

### 2. Principais interpretações
**Lift**
  - Lift está sempre acima de 1.5 em praticamente todas as regras (entre 1.58 e 1.71).

- Quanto maior o Lift, mais forte a relação entre Antecedentes e Consequentes.

- Valores acima de 1.5 são considerados associações fortes ➔ seus resultados são relevantes para análises práticas.

**Confiança**
- A maioria das regras tem confiança muito alta (entre 80% e 100%).

- Confiança de 100% em algumas regras (por exemplo, DELTA_MT → DELTA_LP e DELTA_APROVACAO, DELTA_MT → DELTA_LP) indica certeza total na associação:

  - Sempre que DELTA_MT aumenta, o DELTA_LP também aumenta.

  - Confiança alta reforça que essas relações não são aleatórias.

**Suporte**
- Os suportes variam em torno de 46% a 59%.

- Suporte alto (acima de 10%) mostra que essas relações acontecem com bastante frequência no seu dataset.

**Exemplo: DELTA_IDEB → DELTA_LP tem 58,6% de suporte ➔ mais da metade dos registros seguem essa regra.**

### 3. Padrões encontrados
**Relações frequentes:**
- Muitas combinações entre duas ou três variáveis, formando regras mais complexas e fortes.

**Combinações fortes:**
- Grupos de 2 ou 3 variáveis têm suporte e confiança muito bons.

Ou seja, o comportamento dos indicadores educacionais é interdependente. Isso quer dizer que mudanças em um indicador estão fortemente ligadas a mudanças nos outros.

### 4. Conclusão
- As regras são muito fortes (confiança e lift altos).

- Existem relações diretas entre indicadores de aprovação, IDEB, matemática e língua portuguesa.

- Mudanças em um desses indicadores impactam fortemente os outros

- As regras com alta confiança e bom suporte mostra que seu dataset é consistente e coeso.

# Isolation  Forest

Visualizações Anomalias Rede e Municipio

In [ ]:
from sklearn.ensemble import IsolationForest

# Variáveis numéricas para detectar anomalias
variaveis = ['DELTA_IDEB', 'DELTA_MT', 'DELTA_LP', 'DELTA_APROVACAO']

# Cria ou reseta a coluna 'Anomalia'
df_base['Anomalia'] = 0

# Aplica o Isolation Forest separadamente por REDE
for rede in df_base['REDE'].unique():
    dados_rede = df_base[df_base['REDE'] == rede][variaveis]

    modelo = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    predicoes = modelo.fit_predict(dados_rede)

    indices_rede = df_base[df_base['REDE'] == rede].index
    df_base.loc[indices_rede, 'Anomalia'] = (predicoes == -1).astype(int)


In [ ]:
# Reinicia a coluna de anomalias (caso queira detectar por município agora)
df_base['Anomalia'] = 0

# Aplica o Isolation Forest por cada município individualmente
for municipio in df_base['NO_MUNICIPIO'].unique():
    dados_mun = df_base[df_base['NO_MUNICIPIO'] == municipio][variaveis]

    # Evita executar em municípios com poucos dados
    if len(dados_mun) < 10:
        continue

    modelo = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    preds = modelo.fit_predict(dados_mun)

    indices = df_base[df_base['NO_MUNICIPIO'] == municipio].index
    df_base.loc[indices, 'Anomalia'] = (preds == -1).astype(int)


Exibir as anomalias encontradas (como tabela)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
sns.countplot(data=df_base[df_base['Anomalia'] == 1], x='REDE')
plt.title('🔍 Anomalias por Tipo de Rede de Ensino')
plt.xlabel('Rede')
plt.ylabel('Quantidade de Anomalias')
plt.show()


Gráfico de dispersão para ver as anomalias

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
cores = df_base['Anomalia'].map({0: 'blue', 1: 'red'})  # vermelho = anomalia

plt.scatter(df_base['DELTA_IDEB'], df_base['DELTA_MT'], c=cores, alpha=0.6)
plt.xlabel('Δ IDEB')
plt.ylabel('Δ Matemática')
plt.title('🔍 Anomalias Detectadas por Município (Isolation Forest)')
plt.grid(True)
plt.show()


✅ 3. Contagem de anomalias por tipo de rede (gráfico de barras)
python
Copiar código


In [ ]:
import seaborn as sns

plt.figure(figsize=(10,5))
sns.countplot(data=df_base[df_base['Anomalia'] == 1], x='REDE')
plt.title('Anomalias por Tipo de Rede')
plt.ylabel('Quantidade de Anomalias')
plt.xlabel('Rede de Ensino')
plt.show()


✅ 2. Visualização de Anomalias por MUNICÍPIO (Top 10 com mais anomalias)
python
Copiar código


In [ ]:
anomalias_mun = df_base[df_base['Anomalia'] == 1]['NO_MUNICIPIO'].value_counts().head(10)

plt.figure(figsize=(12,6))
sns.barplot(x=anomalias_mun.values, y=anomalias_mun.index, palette='Reds_r')
plt.title('🏙️ Top 10 Municípios com Mais Anomalias')
plt.xlabel('Número de Anomalias')
plt.ylabel('Município')
plt.show()


# 5 Avaliação (Evaluation)

O estudo realizado oferece uma visão abrangente sobre a evolução do desempenho das escolas públicas brasileiras, especificamente na região Norte, ao longo dos anos. Com base em indicadores educacionais como o IDEB, a taxa de aprovação e o desempenho no SAEB, foi possível estruturar uma análise que identifica padrões distintos entre os municípios.

A abordagem utilizada permitiu compreender melhor a organização dos dados e preparar informações relevantes para modelagem, garantindo que fossem comparáveis ao longo do tempo. A segmentação das escolas com base na evolução dos indicadores possibilitou a identificação de grupos distintos, cada um apresentando características específicas de crescimento ou estagnação.

Os resultados obtidos oferecem insights valiosos para a formulação de políticas públicas direcionadas, permitindo que estratégias possam ser adaptadas conforme as necessidades dos diferentes perfis identificados. Esse estudo contribui para a melhoria do ensino ao destacar boas práticas e possíveis desafios a serem enfrentados na busca por uma educação mais eficiente e equitativa.

### Conclusões e Implicações
- **Principais Descobertas:** Os clusters indicam perfis distintos de desempenho educacional entre as regiões, com algumas redes apresentando evolução consistente.
- **Limitações:** O modelo KMeans não considera outliers, o que pode influenciar a análise.
- **Próximos Passos:** Explorar o impacto de fatores externos (investimentos, políticas públicas) nos indicadores.

## Continuidade

Candidata a fonte junto do Trabalho Não Supervisionado. Separar exploração/modelagem/regras, remover instalação experimental do fluxo principal, corrigir URL em célula de código, distinguir anos iniciais/finais e município/escola; preservar anomalias por rede e município em colunas diferentes (hoje Anomalia é zerada). Validar tratamento de nulos antes de conclusões.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.